In [1]:
import os
import sys
from joblib import Parallel, delayed, cpu_count
from maxent_model import run_maxent_species, init_worker_cache

sys.path.insert(0, 'dataprep')
from species_manifest import load_species_manifest

# Prevent BLAS/OpenMP oversubscription when running many worker processes
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')


'1'

In [2]:
manifest = load_species_manifest()
species_list = [
    {
        'name': row['scientific_name'],
        'group': row['spgroup'],
        'excel_group': row['excel_group'],
    }
    for _, row in manifest.iterrows()
]

# Set True to run only gap-test species (missing param CSVs). False for full production run.
TEST_MODE = False
GAP_TEST_SPECIES_KEYS = [
    'agelaius_phoeniceus',
    'anthus_spragueii',
    'coturnicops_noveboracensis',
    'egretta_caerulea',
    'geothlypis_trichas',
    'melospiza_georgiana',
    'melospiza_melodia',
]

basedir = '/mnt/f/readyparams'
paramdir = os.path.join(basedir, 'param_csvs')
outputdir = os.path.join(basedir, 'modelprep')
aoi = 'mav_counties_4326.parquet'  # within paramdir
os.makedirs(outputdir, exist_ok=True)

TOTAL_CORES = cpu_count()
N_CPUS_PER_SPECIES = 4
N_SPECIES_PARALLEL = max(1, TOTAL_CORES // N_CPUS_PER_SPECIES)
print(f'Parallel config: {N_SPECIES_PARALLEL} species x {N_CPUS_PER_SPECIES} cpus ({TOTAL_CORES} cores)')
print(f'Loaded {len(species_list)} species from Excel manifest')

if TEST_MODE:
    manifest_by_name = manifest.set_index('scientific_name')
    species_list = [
        {
            'name': s,
            'group': manifest_by_name.loc[s, 'spgroup'],
            'excel_group': manifest_by_name.loc[s, 'excel_group'],
        }
        for s in GAP_TEST_SPECIES_KEYS
    ]
    print(f'TEST_MODE: {len(species_list)} gap-test species')


Parallel config: 10 species x 4 cpus (40 cores)


In [3]:
for s in species_list:
    if isinstance(s.get('name'), str):
        s['name'] = s['name'].replace(' ', '_').lower()


def run_one(species):
    try:
        print('run', species['name'])
        results = run_maxent_species(
            sp=species['name'],
            spgroup=species['group'],
            parambasedir=paramdir,
            baseoutputdir=basedir,
            aoi_filename=aoi,
            excel_group=species.get('excel_group', 'birds'),
            n_cpus=N_CPUS_PER_SPECIES,
            batch_mode=True,
            inner_parallel=False,
            selection_metric='mean_auc',
        )
        return (species, 'run complete', results)
    except Exception as exc:
        return (species, 'fail', exc)


In [4]:
completed = Parallel(
    n_jobs=N_SPECIES_PARALLEL,
    prefer='processes',
    verbose=1,
    initializer=init_worker_cache,
    initargs=(paramdir, aoi),
)(delayed(run_one)(species_list[i]) for i in range(len(species_list)))

failed = [result for result in completed if result[1] == 'fail']
if failed:
    print(f'Retrying {len(failed)} failed species sequentially...')
    init_worker_cache(paramdir, aoi)
    retry_results = [run_one(result[0]) for result in failed]
    completed = [
        result for result in completed if result[1] != 'fail'
    ] + retry_results


[Parallel(n_jobs=10)]: Using backend LokyBackend with 10 concurrent workers.
/home/mike/miniforge3/envs/rapids-25.10/lib/python3.13/site-packages/geopandas/array.py:408: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(
/home/mike/miniforge3/envs/rapids-25.10/lib/python3.13/site-packages/geopandas/array.py:408: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(
/home/mike/miniforge3/envs/rapids-25.10/lib/python3.13/site-packages/geopandas/array.py:408: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(
/home/mike/miniforge3/e

run agelaius_phoeniceus
/mnt/f/readyparams/ppp_paramsoutput/agelaius_phoeniceus
run calidris_pusilla
/mnt/f/readyparams/ppp_paramsoutput/calidris_pusilla
Reading: /mnt/f/readyparams/param_csvs/calidris_pusilla_subset0.csv
Reading: /mnt/f/readyparams/param_csvs/calidris_pusilla_subset1.csv
Reading: /mnt/f/readyparams/param_csvs/background_avian.csv
Total presence points: 504
Total background points: 2171
Fold type: GeographicKFold
Selected beta_multiplier=4.0 (CV mean_auc=0.9030)
Final tuned model (beta_multiplier=4.0)
AUC=0.9620  Precision=0.8455  Recall=0.7817  F1=0.8124
Best threshold=0.4867  Log-loss=0.1982  Prevalence=0.1884
Tuned final MaxEnt (PPP-equivalent) model saved, with diagnostics & importance.


/home/mike/miniforge3/envs/rapids-25.10/lib/python3.13/site-packages/geopandas/array.py:408: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(
/home/mike/miniforge3/envs/rapids-25.10/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
/home/mike/miniforge3/envs/rapids-25.10/lib/python3.13/site-packages/geopandas/array.py:408: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(
/home/mike/miniforge3/envs/rapids-25.10/lib/python3.13/site-packages/geopandas/array.py:408: UserWarning: Geometry is in a 

Retrying 8 failed species sequentially...
run agelaius_phoeniceus
/mnt/f/readyparams/ppp_paramsoutput/agelaius_phoeniceus
run anthus_spragueii
/mnt/f/readyparams/ppp_paramsoutput/anthus_spragueii
Reading: /mnt/f/readyparams/param_csvs/anthus_spragueii_subset0.csv
Reading: /mnt/f/readyparams/param_csvs/anthus_spragueii_subset1.csv
Reading: /mnt/f/readyparams/param_csvs/background_avian.csv


/home/mike/miniforge3/envs/rapids-25.10/lib/python3.13/site-packages/geopandas/array.py:408: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(


Total presence points: 16
Total background points: 1407
run centronyx_henslowii
/mnt/f/readyparams/ppp_paramsoutput/centronyx_henslowii
Reading: /mnt/f/readyparams/param_csvs/centronyx_henslowii_subset0.csv
Reading: /mnt/f/readyparams/param_csvs/centronyx_henslowii_subset1.csv


/home/mike/miniforge3/envs/rapids-25.10/lib/python3.13/site-packages/geopandas/array.py:408: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(


Total presence points: 16
Total background points: 1734
run coturnicops_noveboracensis
/mnt/f/readyparams/ppp_paramsoutput/coturnicops_noveboracensis
Reading: /mnt/f/readyparams/param_csvs/coturnicops_noveboracensis_subset0.csv
run egretta_caerulea
/mnt/f/readyparams/ppp_paramsoutput/egretta_caerulea
run hyla_versicolor
/mnt/f/readyparams/ppp_paramsoutput/hyla_versicolor
Reading: /mnt/f/readyparams/param_csvs/hyla_versicolor_subset0.csv
Reading: /mnt/f/readyparams/param_csvs/hyla_versicolor_subset1.csv
Reading: /mnt/f/readyparams/param_csvs/background_herp.csv


/home/mike/miniforge3/envs/rapids-25.10/lib/python3.13/site-packages/geopandas/array.py:408: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(


Total presence points: 10
Total background points: 2626
run lithobates_palustris
/mnt/f/readyparams/ppp_paramsoutput/lithobates_palustris
Reading: /mnt/f/readyparams/param_csvs/lithobates_palustris_subset0.csv
run macrochelys_temmincki
/mnt/f/readyparams/ppp_paramsoutput/macrochelys_temmincki


In [9]:
print('## Failed species ##')
print('')
for result in completed:
    if result[1] == 'fail':
        print(result[0])
        print(result[2])


## Failed species ##

{'name': 'agelaius_phoeniceus', 'group': 'avian'}
cannot concat empty list
{'name': 'anthus_spragueii', 'group': 'avian'}
Not enough presence records
{'name': 'centronyx_henslowii', 'group': 'avian'}
Not enough presence records
{'name': 'coturnicops_noveboracensis', 'group': 'avian'}
Number of dimensions is greater than number of samples. This results in a singular data covariance matrix, which cannot be treated using the algorithms implemented in `gaussian_kde`. Note that `gaussian_kde` interprets each *column* of `dataset` to be a point; consider transposing the input to `dataset`.
{'name': 'egretta_caerulea', 'group': 'avian'}
cannot concat empty list
{'name': 'hyla_versicolor', 'group': 'herp'}
Not enough presence records
{'name': 'lithobates_palustris', 'group': 'herp'}
'a' and 'p' must have same size
{'name': 'macrochelys_temmincki', 'group': 'herp'}
cannot concat empty list
